# RLO ImageNet Training - 8x B200

**稳定版本** - 避免所有multiprocessing问题

In [ ]:
# =============================================================================
# CELL 1: 设置
# =============================================================================

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import time
import math
import random
import json
import gc
import warnings
from pathlib import Path
from typing import Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# GPU优化
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 路径
DATA_ROOT = Path("/blue/wdixon/wang.yixuan/lypcdf/imagenet_folder")
SAVE_DIR = Path("./results")
SAVE_DIR.mkdir(exist_ok=True)

# GPU检查
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

In [ ]:
# =============================================================================
# CELL 2: 你的RLO优化器 (不修改)
# =============================================================================

class RLO(Optimizer):
    """Riemannian Lyapunov Optimizer - 你的原始实现"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.1,
                 belief_coef=0.1, eps=1e-8):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay,
                        belief_coef=belief_coef, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            wd = group["weight_decay"]
            beta1, beta2 = group["betas"]
            belief = group["belief_coef"]
            eps = group["eps"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                g = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["exp_avg"] = torch.zeros_like(p)

                m = state["exp_avg"]

                # Decoupled weight decay
                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)

                # Compute direction
                c = beta1 * m + (1.0 - beta1) * g
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                d = c.sign() + belief * (delta / delta_norm)

                # Update
                p.add_(d, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=(1.0 - beta2))

        return loss


class RLO_LambdaA(Optimizer):
    """RLO with Lambda-A preconditioning - 你的原始实现"""
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, beta3=0.999,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, beta3=beta3,
                        weight_decay=weight_decay, lambda_b=lambda_b,
                        eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        all_smooth_pre = []
        all_belief = []
        all_params = []

        for group in self.param_groups:
            eps = group["eps"]
            gamma = group["gamma"]
            beta1 = group["beta1"]
            beta2 = group["beta2"]
            beta3 = group["beta3"]
            lambda_b = group["lambda_b"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                g = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["s"] = torch.zeros_like(p)

                m = state["m"]
                s = state["s"]

                # Update second moment
                s.mul_(beta3).addcmul_(g, g, value=(1.0 - beta3))

                # Compute smooth direction
                c = beta1 * m + (1.0 - beta1) * g
                smooth = torch.tanh(gamma * c)
                smooth_pre = smooth / (s.sqrt() + eps)

                # Belief correction
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                belief = lambda_b * (delta / delta_norm)

                all_smooth_pre.append(smooth_pre)
                all_belief.append(belief)
                all_params.append((p, group))

        if not all_params:
            return loss

        # Global normalization
        s_norm = sum((sp * sp).sum() for sp in all_smooth_pre).sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm

        for (p, group), sp, b in zip(all_params, all_smooth_pre, all_belief):
            lr = group["lr"]
            wd = group["weight_decay"]
            beta2 = group["beta2"]

            d = scale * sp + b
            state = self.state[p]

            if wd != 0.0:
                p.mul_(1.0 - lr * wd)

            p.add_(d, alpha=-lr)
            state["m"].mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))

        return loss


class Lion(Optimizer):
    """Lion optimizer (Google)"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])

                g = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)

                m = state['exp_avg']
                beta1, beta2 = group['betas']

                update = (beta1 * m + (1 - beta1) * g).sign_()
                p.add_(update, alpha=-group['lr'])
                m.mul_(beta2).add_(g, alpha=1 - beta2)


print("优化器加载完成: RLO, RLO_LambdaA, Lion, AdamW")

In [ ]:
# =============================================================================
# CELL 3: 数据加载 (num_workers=0 避免multiprocessing问题)
# =============================================================================

def create_dataloaders(batch_size=512, use_augment=False):
    """
    创建数据加载器。
    
    关键: num_workers=0 避免所有multiprocessing问题！
    虽然慢一点，但稳定。
    """
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)
    
    if use_augment:
        train_transform = T.Compose([
            T.RandomResizedCrop(224),
            T.RandomHorizontalFlip(),
            T.RandAugment(num_ops=2, magnitude=9),
            T.ToTensor(),
            T.Normalize(MEAN, STD),
        ])
    else:
        train_transform = T.Compose([
            T.RandomResizedCrop(224),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(MEAN, STD),
        ])
    
    val_transform = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
    
    train_set = ImageFolder(DATA_ROOT / 'train', train_transform)
    val_set = ImageFolder(DATA_ROOT / 'val', val_transform)
    
    # num_workers=0: 主进程加载数据，避免multiprocessing问题
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size*2, shuffle=False,
        num_workers=0, pin_memory=True
    )
    
    print(f"数据集: Train={len(train_set)}, Val={len(val_set)}")
    print(f"Batch size: {batch_size}, Batches/epoch: {len(train_loader)}")
    
    return train_loader, val_loader


# 测试数据加载
print("测试数据加载...")
_loader, _ = create_dataloaders(batch_size=64)
for x, y in _loader:
    print(f"✓ 测试batch: {x.shape}, {y.shape}")
    break
del _loader
print("数据加载正常!")

In [ ]:
# =============================================================================
# CELL 4: 模型
# =============================================================================

def create_resnet50():
    from torchvision.models import resnet50
    return resnet50(weights=None, num_classes=1000)


class ViT(nn.Module):
    """Vision Transformer"""
    def __init__(self, dim=384, depth=12, heads=6, mlp_ratio=4, num_classes=1000):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, 16, 16)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))
        
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim, nhead=heads, dim_feedforward=dim*mlp_ratio,
                dropout=0.0, activation='gelu', batch_first=True, norm_first=True
            ) for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # B, 196, dim
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)  # B, 197, dim
        x = x + self.pos_embed
        
        for blk in self.blocks:
            x = blk(x)
        
        return self.head(self.norm(x[:, 0]))


def create_vit_s():
    return ViT(dim=384, depth=12, heads=6)

def create_vit_b():
    return ViT(dim=768, depth=12, heads=12)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"ResNet-50: {count_params(create_resnet50())/1e6:.1f}M params")
print(f"ViT-S/16: {count_params(create_vit_s())/1e6:.1f}M params")
print(f"ViT-B/16: {count_params(create_vit_b())/1e6:.1f}M params")

In [ ]:
# =============================================================================
# CELL 5: 训练函数
# =============================================================================

@torch.no_grad()
def evaluate(model, loader, device):
    """评估模型"""
    model.eval()
    correct, total = 0, 0
    for x, y in tqdm(loader, desc="Eval", leave=False):
        x, y = x.to(device), y.to(device)
        with torch.autocast('cuda', torch.bfloat16):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


def train_one_model(
    model_fn,
    optimizer_name: str,
    epochs: int,
    batch_size: int,
    lr: float,
    wd: float,
    warmup_epochs: int = 5,
    use_augment: bool = False,
    label_smoothing: float = 0.0,
):
    """
    训练单个模型。
    使用DataParallel实现多GPU。
    """
    set_seed(42)
    gc.collect()
    torch.cuda.empty_cache()
    
    device = torch.device('cuda:0')
    num_gpus = torch.cuda.device_count()
    
    # 数据
    train_loader, val_loader = create_dataloaders(batch_size, use_augment)
    
    # 模型
    model = model_fn().to(device)
    if num_gpus > 1:
        model = nn.DataParallel(model)
    
    # 优化器
    if optimizer_name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif optimizer_name == 'lion':
        opt = Lion(model.parameters(), lr=lr, weight_decay=wd)
    elif optimizer_name == 'rlo':
        opt = RLO(model.parameters(), lr=lr, weight_decay=wd)
    elif optimizer_name == 'rlo_lambda_a':
        opt = RLO_LambdaA(model.parameters(), lr=lr, weight_decay=wd)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")
    
    # 学习率调度
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    
    def get_lr(step):
        if step < warmup_steps:
            return lr * step / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return lr * 0.5 * (1 + math.cos(math.pi * progress))
    
    # 损失函数
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    scaler = torch.cuda.amp.GradScaler()
    
    # 记录
    history = {'train_acc': [], 'val_acc': [], 'loss': [], 'throughput': []}
    best_acc = 0.0
    global_step = 0
    
    print(f"\n{'='*60}")
    print(f"训练: {optimizer_name}")
    print(f"GPUs: {num_gpus}, Batch: {batch_size}, LR: {lr}, WD: {wd}")
    print(f"Epochs: {epochs}, Steps/epoch: {len(train_loader)}")
    print(f"{'='*60}")
    
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        correct, total, running_loss = 0, 0, 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            # 更新学习率
            current_lr = get_lr(global_step)
            for g in opt.param_groups:
                g['lr'] = current_lr
            
            opt.zero_grad(set_to_none=True)
            
            with torch.autocast('cuda', torch.bfloat16):
                out = model(x)
                loss = criterion(out, y)
            
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
            running_loss += loss.item()
            global_step += 1
            
            if global_step % 100 == 0:
                pbar.set_postfix({
                    'loss': f'{running_loss/100:.4f}',
                    'acc': f'{100*correct/total:.1f}%',
                    'lr': f'{current_lr:.2e}'
                })
                running_loss = 0.0
        
        # Epoch结束
        epoch_time = time.time() - t0
        throughput = len(train_loader) * batch_size / epoch_time
        train_acc = 100 * correct / total
        
        # 验证
        val_model = model.module if hasattr(model, 'module') else model
        val_acc = evaluate(val_model, val_loader, device)
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['throughput'].append(throughput)
        
        is_best = val_acc > best_acc
        best_acc = max(val_acc, best_acc)
        
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Val={val_acc:.1f}%{' *' if is_best else ''}, "
              f"Throughput={throughput:.0f} img/s")
        
        # 保存最佳模型
        if is_best:
            torch.save(val_model.state_dict(), SAVE_DIR / f"{optimizer_name}_best.pt")
    
    # 清理
    del model, opt, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'optimizer': optimizer_name,
        'best_acc': best_acc,
        'final_acc': history['val_acc'][-1],
        'avg_throughput': np.mean(history['throughput']),
        'history': history,
    }


print("训练函数准备完成")

In [ ]:
# =============================================================================
# CELL 6: 画图函数
# =============================================================================

def plot_results(results: Dict, title: str, save_path=None):
    """画训练曲线"""
    colors = {
        'adamw': '#1f77b4',
        'lion': '#ff7f0e', 
        'rlo': '#2ca02c',
        'rlo_lambda_a': '#d62728',
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for name, res in results.items():
        h = res['history']
        c = colors.get(name, 'gray')
        axes[0].plot(h['train_acc'], label=name, color=c, lw=2)
        axes[1].plot(h['val_acc'], label=name, color=c, lw=2)
        axes[2].plot(h['throughput'], label=name, color=c, lw=2)
    
    axes[0].set_title('Train Accuracy'); axes[0].set_ylabel('%')
    axes[1].set_title('Val Accuracy'); axes[1].set_ylabel('%')
    axes[2].set_title('Throughput'); axes[2].set_ylabel('img/s')
    
    for ax in axes:
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('Epoch')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def print_table(results: Dict, title: str):
    """打印结果表格"""
    print(f"\n{'='*55}")
    print(f"{title}")
    print(f"{'='*55}")
    print(f"{'Optimizer':<20} {'Best':>10} {'Final':>10} {'img/s':>12}")
    print(f"{'-'*55}")
    for name, res in sorted(results.items(), key=lambda x: x[1]['best_acc'], reverse=True):
        print(f"{name:<20} {res['best_acc']:>9.2f}% {res['final_acc']:>9.2f}% {res['avg_throughput']:>11.0f}")


print("画图函数准备完成")

In [ ]:
# =============================================================================
# CELL 7: Quick Test (3 epochs验证一切正常)
# =============================================================================

def quick_test():
    """快速测试，验证训练流程正常"""
    print("\n" + "="*60)
    print("QUICK TEST: 3 epochs验证训练流程")
    print("="*60)
    
    result = train_one_model(
        model_fn=create_resnet50,
        optimizer_name='adamw',
        epochs=3,
        batch_size=512,  # 每GPU 512 * 8 = 4096 total
        lr=1e-3,
        wd=0.05,
        warmup_epochs=1,
    )
    
    print(f"\n✓ Quick test完成!")
    print(f"  Best acc: {result['best_acc']:.2f}%")
    print(f"  Throughput: {result['avg_throughput']:.0f} img/s")
    
    return result

# 运行quick test
quick_result = quick_test()

In [ ]:
# =============================================================================
# CELL 8: 实验4.1 - ResNet-50 (90 epochs)
# =============================================================================

def run_resnet50_experiments():
    """
    ResNet-50 on ImageNet
    Following LION paper settings
    """
    print("\n" + "#"*60)
    print("# EXPERIMENT: ResNet-50 on ImageNet")
    print("#"*60)
    
    # 超参数 (from LION paper)
    configs = {
        'adamw': {'lr': 1e-3, 'wd': 0.05},
        'lion': {'lr': 1e-4, 'wd': 0.5},       # 10x smaller lr, 10x larger wd
        'rlo': {'lr': 1e-4, 'wd': 0.5},
        'rlo_lambda_a': {'lr': 1e-4, 'wd': 0.5},
    }
    
    results = {}
    
    for opt_name, cfg in configs.items():
        print(f"\n>>> 训练 {opt_name}...")
        
        try:
            result = train_one_model(
                model_fn=create_resnet50,
                optimizer_name=opt_name,
                epochs=90,
                batch_size=512,  # 8 GPU * 512 = 4096 effective
                lr=cfg['lr'],
                wd=cfg['wd'],
                warmup_epochs=5,
                use_augment=False,
                label_smoothing=0.0,
            )
            results[opt_name] = result
            
            # 每完成一个优化器就画图
            print_table(results, "ResNet-50 Progress")
            plot_results(results, "ResNet-50 on ImageNet", SAVE_DIR / "resnet50_progress.png")
            
            # 保存结果
            with open(SAVE_DIR / "resnet50_results.json", 'w') as f:
                json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'history'} 
                          for k, v in results.items()}, f, indent=2)
                
        except Exception as e:
            print(f"!!! {opt_name} 失败: {e}")
            import traceback
            traceback.print_exc()
    
    # 最终结果
    print_table(results, "ResNet-50 Final Results")
    plot_results(results, "ResNet-50 on ImageNet (Final)", SAVE_DIR / "resnet50_final.png")
    
    return results

# 运行实验
# resnet_results = run_resnet50_experiments()

In [ ]:
# =============================================================================
# CELL 9: 实验4.1 - ViT-S/16 (300 epochs)
# =============================================================================

def run_vit_s_experiments():
    """
    ViT-S/16 on ImageNet
    With RandAugment + Mixup
    """
    print("\n" + "#"*60)
    print("# EXPERIMENT: ViT-S/16 on ImageNet")
    print("#"*60)
    
    configs = {
        'adamw': {'lr': 1e-3, 'wd': 0.1},
        'lion': {'lr': 1e-4, 'wd': 1.0},
        'rlo': {'lr': 1e-4, 'wd': 1.0},
        'rlo_lambda_a': {'lr': 1e-4, 'wd': 1.0},
    }
    
    results = {}
    
    for opt_name, cfg in configs.items():
        print(f"\n>>> 训练 {opt_name}...")
        
        try:
            result = train_one_model(
                model_fn=create_vit_s,
                optimizer_name=opt_name,
                epochs=300,
                batch_size=256,  # 8 GPU * 256 = 2048 effective
                lr=cfg['lr'],
                wd=cfg['wd'],
                warmup_epochs=30,
                use_augment=True,
                label_smoothing=0.1,
            )
            results[opt_name] = result
            
            print_table(results, "ViT-S/16 Progress")
            plot_results(results, "ViT-S/16 on ImageNet", SAVE_DIR / "vit_s_progress.png")
            
            with open(SAVE_DIR / "vit_s_results.json", 'w') as f:
                json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'history'} 
                          for k, v in results.items()}, f, indent=2)
                
        except Exception as e:
            print(f"!!! {opt_name} 失败: {e}")
    
    print_table(results, "ViT-S/16 Final Results")
    plot_results(results, "ViT-S/16 on ImageNet (Final)", SAVE_DIR / "vit_s_final.png")
    
    return results

# 运行实验
# vit_s_results = run_vit_s_experiments()

In [ ]:
# =============================================================================
# CELL 10: 实验4.1 - ViT-B/16 (300 epochs)
# =============================================================================

def run_vit_b_experiments():
    """
    ViT-B/16 on ImageNet
    """
    print("\n" + "#"*60)
    print("# EXPERIMENT: ViT-B/16 on ImageNet")
    print("#"*60)
    
    configs = {
        'adamw': {'lr': 1e-3, 'wd': 0.1},
        'lion': {'lr': 1e-4, 'wd': 1.0},
        'rlo': {'lr': 1e-4, 'wd': 1.0},
        'rlo_lambda_a': {'lr': 1e-4, 'wd': 1.0},
    }
    
    results = {}
    
    for opt_name, cfg in configs.items():
        print(f"\n>>> 训练 {opt_name}...")
        
        try:
            result = train_one_model(
                model_fn=create_vit_b,
                optimizer_name=opt_name,
                epochs=300,
                batch_size=128,  # 8 GPU * 128 = 1024 effective
                lr=cfg['lr'],
                wd=cfg['wd'],
                warmup_epochs=30,
                use_augment=True,
                label_smoothing=0.1,
            )
            results[opt_name] = result
            
            print_table(results, "ViT-B/16 Progress")
            plot_results(results, "ViT-B/16 on ImageNet", SAVE_DIR / "vit_b_progress.png")
            
            with open(SAVE_DIR / "vit_b_results.json", 'w') as f:
                json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'history'} 
                          for k, v in results.items()}, f, indent=2)
                
        except Exception as e:
            print(f"!!! {opt_name} 失败: {e}")
    
    print_table(results, "ViT-B/16 Final Results")
    plot_results(results, "ViT-B/16 on ImageNet (Final)", SAVE_DIR / "vit_b_final.png")
    
    return results

# 运行实验
# vit_b_results = run_vit_b_experiments()

In [ ]:
# =============================================================================
# CELL 11: 运行所有实验
# =============================================================================

def run_all_experiments():
    """运行所有实验"""
    all_results = {}
    
    print("\n" + "="*70)
    print("开始所有实验...")
    print("="*70)
    
    # ResNet-50
    all_results['resnet50'] = run_resnet50_experiments()
    
    # ViT-S/16
    all_results['vit_s'] = run_vit_s_experiments()
    
    # ViT-B/16
    all_results['vit_b'] = run_vit_b_experiments()
    
    print("\n" + "="*70)
    print("所有实验完成!")
    print("="*70)
    
    # 保存所有结果
    with open(SAVE_DIR / "all_results.json", 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    return all_results

# 运行所有实验
# all_results = run_all_experiments()

In [ ]:
print("""
================================================================================
RLO ImageNet Training - 稳定版本
================================================================================

关键设计决策:
  1. num_workers=0: 避免所有multiprocessing问题
  2. GradScaler: 稳定的混合精度训练
  3. 梯度裁剪: 防止梯度爆炸
  4. 正确的学习率: AdamW用1e-3, Lion/RLO用1e-4

使用方法:

1. 运行Cell 1-6加载所有组件

2. 运行Cell 7的quick_test()验证一切正常:
   quick_result = quick_test()

3. 运行实验:
   - ResNet-50: resnet_results = run_resnet50_experiments()  # ~15h
   - ViT-S/16:  vit_s_results = run_vit_s_experiments()      # ~40h
   - ViT-B/16:  vit_b_results = run_vit_b_experiments()      # ~60h
   
   或者一次运行所有:
   all_results = run_all_experiments()

结果保存在: ./results/

================================================================================
""")